# Does `get_structure` recover a known complex from sequence alone?

And - the question that actually decides how the rest of the pipeline is
weighted - **do the confidence metrics predict when it fails?**

Protocol, per target: take the deposited structure as ground truth, read the
three chain sequences off it, throw the coordinates away, predict the complex
from the sequences, compare.

Two tiers, kept apart:

| Tier | What it is | What a good result means |
|---|---|---|
| **smoke** | 1VFB, 1N8Z, 3HFM | The plumbing works. Nothing else: these are memorized - they are in every predictor's training set. |
| **honest** | selected from SAbDab, deposited after the predictor's training cutoff | Something about generalisation. |

Smoke-tier numbers are reported in their own table and **never averaged with
the honest tier**.

Everything is cached through the pipeline's own content-addressed cache, so a
rerun is free. With no CUDA GPU the notebook switches to the fixture predictor
and synthetic targets so it still executes end to end - and says so, loudly,
because those numbers mean nothing.

In [ ]:
# --- parameters -------------------------------------------------------------

PREDICTOR = "boltz2"      # open weights, runs locally. "alphafold3" where access allows.
NUM_SEEDS = 15            # 15-20 per target; the ensemble is the point
LIMIT = 6                 # honest-tier targets
K = 10                    # top-k for the ranking metrics
OUTPUT_DIR = "runs/structure_validation"

# A kept local copy of the SAbDab summary TSV makes this reproducible: the file
# grows, so a rerun six months from now selects a different set. None downloads it.
SABDAB = None

# The predictor's own documented training cutoff. This is a parameter, not a
# constant, because getting it wrong silently puts training data into the
# honest tier and turns the whole exercise into a memorisation test.
#   AlphaFold3: 2021-09-30 (published in the paper's methods) - filled in below.
#   Boltz-2, Chai-1: read it from the model card or repo and set it here.
#   No documented cutoff: benchmark.cutoff_from_release("<release date>").
# Stricter is always safe - it only shrinks the pool.
TRAINING_CUTOFF = None

FORCE_DUMMY = None        # None -> automatic: fixtures when no CUDA GPU is visible

In [ ]:
import warnings
from pathlib import Path

import pandas as pd

from design_pipeline import benchmark, confidence, metrics
from design_pipeline.adapters._folding import gpus

DUMMY = (not gpus()) if FORCE_DUMMY is None else bool(FORCE_DUMMY)
predictor = "dummy_predictor" if DUMMY else PREDICTOR
out = Path(OUTPUT_DIR + ("_dummy" if DUMMY else ""))

cutoff, cutoff_source, verified = benchmark.training_cutoff(predictor)
if TRAINING_CUTOFF:
    cutoff, cutoff_source, verified = TRAINING_CUTOFF, "set in this notebook", True

print(f"predictor:       {predictor}")
print(f"seeds/target:    {NUM_SEEDS}")
print(f"output:          {out}")
print(f"training cutoff: {cutoff or 'UNKNOWN'}")
print(f"  source:        {cutoff_source}")
print(f"  verified:      {verified}")
if DUMMY:
    print()
    print("!! no CUDA GPU visible: running the fixture predictor on synthetic")
    print("!! targets. Every number below is meaningless. This mode exists so")
    print("!! the notebook executes in CI, not so it produces a result.")

## Smoke tier

Three complexes everybody uses. **Memorized - plumbing check only.**

The list is hardcoded and not trusted: each entry is checked against RCSB
before use - resolution present and under 3.0 A, at least three protein
polymer entities, and the chains named here actually present with an antigen
that is not an antibody chain and not a peptide. A failed check prints and
drops the entry rather than proceeding.

In [ ]:
if DUMMY:
    smoke, dropped = [], []
    print("fixture mode: skipping the RCSB validation and the smoke tier")
else:
    smoke, dropped = benchmark.validate_smoke_tier()

for target, reasons in dropped:
    print(f"DROPPED {target.pdb_id}: " + "; ".join(reasons))

if smoke:
    display(
        pd.DataFrame(
            [
                {
                    "pdb_id": t.pdb_id,
                    "H": t.heavy,
                    "L": t.light,
                    "antigen": t.antigen,
                    "resolution": t.resolution,
                    "released": t.date,
                    "note": t.note,
                }
                for t in smoke
            ]
        )
    )

## Honest tier

Raw RCSB attribute queries detect antibodies badly, so the selection comes from
SAbDab's summary file instead. The filters run in a fixed order and the
surviving count is printed after each, so the set is reproducible and
re-runnable as SAbDab grows:

1. a protein antigen - drops peptides, haptens, nucleic acids and unbound entries
2. both a heavy and a light chain - drops nanobodies and single-domain entries
3. an antigen chain distinct from both
4. X-ray diffraction at 3.0 A or better
5. deposited after the training cutoff
6. one entry per antigen cluster at 70% sequence identity, best resolution kept

Step 6 is not optional. Without it the set fills with near-identical
SARS-CoV-2 RBD complexes and six entries are one data point wearing six hats.

If SAbDab is unreachable this fails and names the URL. There is no fallback to
a hardcoded list: a silently stale honest tier is worse than none.

In [ ]:
sources: dict[str, Path] = {}

if DUMMY:
    fixtures = benchmark.fixture_targets(out / "fixtures", 3)
    honest = [target for target, _ in fixtures]
    sources = {target.pdb_id: path for target, path in fixtures}
    print("fixture mode: three synthetic complexes instead of a SAbDab selection.")
    print("They have a real interface and real chain relabelling, and nothing else real.")
elif cutoff is None:
    raise RuntimeError(
        f"no training cutoff known for {predictor!r}, so an honest tier cannot "
        f"be built: anything deposited before it may be training data. Read the "
        f"cutoff from the model card and set TRAINING_CUTOFF above, or use "
        f"benchmark.cutoff_from_release('<release date>')."
    )
else:
    frame = benchmark.read_sabdab(SABDAB or benchmark.SABDAB_SUMMARY)
    honest, log = benchmark.select_sabdab(
        frame,
        cutoff,
        limit=LIMIT,
        sequences=benchmark.fetch_sequences,
    )
    print("\n".join(log))

display(
    pd.DataFrame(
        [
            {
                "pdb_id": t.pdb_id,
                "H": t.heavy,
                "L": t.light,
                "antigen": t.antigen,
                "deposited": t.date,
                "resolution": t.resolution,
            }
            for t in honest
        ]
    )
)

## Run

The decoy control pairs each antibody with **another target's antigen** - a
non-cognate pair of two real proteins, which is the control the published
benchmarks used. A shuffled sequence would be a weaker control: it is not a
protein at all, so a low score could just mean "that is not foldable".

Predictions, references and confidence all land in the content-addressed
cache, so this cell is free on a rerun.

In [ ]:
targets = [*smoke, *honest]
if not targets:
    raise RuntimeError("no targets survived selection; nothing to evaluate")

# The three resolved chain sequences per target, read off the deposited
# structure. Needed before the loop so each target can borrow the next one's
# antigen as its decoy.
antigens: dict[str, str] = {}
for target in targets:
    _, (heavy, light, antigen) = benchmark.reference_structure(
        target, output_dir=out, reference_source=sources.get(target.pdb_id)
    )
    antigens[target.pdb_id] = antigen
    print(f"{target.pdb_id}: H={len(heavy)} L={len(light)} antigen={len(antigen)} residues")

order = list(antigens)
rows = []
for index, target in enumerate(targets):
    decoy = antigens[order[(index + 1) % len(order)]] if len(order) > 1 else None
    with warnings.catch_warnings(record=True) as raised:
        warnings.simplefilter("always")
        row = benchmark.evaluate_target(
            target,
            output_dir=out,
            model_name=predictor,
            num_seeds=NUM_SEEDS,
            reference_source=sources.get(target.pdb_id),
            decoy_antigen_seq=decoy,
        )
    row["warnings"] = "; ".join(str(w.message)[:60] for w in raised)
    rows.append(row)
    print(f"  {target.pdb_id} done")

results = pd.DataFrame(rows)

## Results

One table. `epitope_recall` is the column to read first: a prediction can score
passably on DockQ while putting the paratope on the wrong patch of the antigen,
and that silently poisons every structure-based scorer downstream with no sign
that anything is wrong.

DockQ bands: 0.23 acceptable, 0.49 medium, 0.80 high. The column is NaN when
the DockQ package is not installed (`pip install 'design-pipeline[metrics]'`);
`interface_rmsd` and `epitope_recall` need nothing extra.

In [ ]:
COLUMNS = {
    "pdb_id": "target",
    "date": "deposited",
    "tier": "tier",
    "dockq": "DockQ",
    "dockq_quality": "band",
    "interface_rmsd": "iRMSD",
    "epitope_recall": "epitope recall",
    "confidence_iptm": "ipTM",
    "confidence_dominant_cluster_fraction": "dominant cluster",
    "confidence_decoy_gap": "decoy ipTM gap",
}
table = results.reindex(columns=list(COLUMNS)).rename(columns=COLUMNS)

for tier, group in table.groupby("tier", sort=True):
    label = {
        "smoke": "SMOKE TIER - memorized, plumbing check only",
        "honest": "HONEST TIER - deposited after the training cutoff",
        "fixture": "FIXTURE - synthetic, meaningless numbers",
    }.get(tier, tier)
    print(label)
    display(group.drop(columns="tier").round(3))

## The calibration question

This is the point of the notebook. Two candidate signals for "do not trust this
structure":

* **ipTM** - the model grading its own work. Cheap, and these models are
  documented to score deliberately non-cognate antibody-antigen pairs
  confidently.
* **dominant cluster fraction** - how many of the seeds agree on one interface.
  Not self-assessment, which is why it is worth more.

If either correlates with accuracy on the honest tier, it can steer scorer
weights. If neither does, `recommend_weights` is guessing and the structure-free
scorer columns deserve more weight by default.

The accuracy axis is DockQ where available and epitope recall otherwise, and
the correlation is computed on the **honest tier only**.

In [ ]:
honest_rows = results[results["tier"] == "honest"]
scoring_rows = honest_rows if len(honest_rows) else results
if not len(honest_rows):
    print("no honest tier in this run: correlating over whatever rows exist,")
    print("which for fixtures or smoke-only means the number says nothing.")

accuracy_column = (
    "dockq"
    if "dockq" in scoring_rows and scoring_rows["dockq"].notna().any()
    else "epitope_recall"
)
accuracy = scoring_rows[accuracy_column]
print(f"accuracy axis: {accuracy_column}  (n={len(scoring_rows)})")
print()

signals = {
    "ipTM": "confidence_iptm",
    "dominant cluster fraction": "confidence_dominant_cluster_fraction",
}
correlations = {}
for label, column in signals.items():
    if column not in scoring_rows:
        continue
    rho = metrics.spearman(scoring_rows[column], accuracy)
    correlations[label] = rho
    print(f"  Spearman({label}, {accuracy_column}) = {rho:.3f}" if rho == rho
          else f"  Spearman({label}, {accuracy_column}) = n/a (no variance, or too few points)")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
    print("matplotlib is not installed, so the two plots are skipped.")
    print("The correlations above are the result; the plots only show their shape.")

if plt is not None and len(scoring_rows):
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
    for axis, (label, column) in zip(axes, signals.items()):
        if column not in scoring_rows:
            continue
        axis.scatter(scoring_rows[column], accuracy, s=60)
        for _, row in scoring_rows.iterrows():
            axis.annotate(
                row["pdb_id"],
                (row[column], row[accuracy_column]),
                fontsize=8,
                xytext=(4, 4),
                textcoords="offset points",
            )
        rho = correlations.get(label, float("nan"))
        axis.set_xlabel(label)
        axis.set_title(f"rho = {rho:.2f}" if rho == rho else "rho = n/a")
    if accuracy_column == "dockq":
        for axis in axes:
            for level, name in ((0.23, "acceptable"), (0.49, "medium"), (0.80, "high")):
                axis.axhline(level, linestyle=":", linewidth=1)
                axis.annotate(name, (axis.get_xlim()[0], level), fontsize=7, va="bottom")
    axes[0].set_ylabel(accuracy_column)
    figure.suptitle(
        f"Does confidence track accuracy? {predictor}, "
        f"{'fixtures' if DUMMY else 'honest tier'}"
    )
    figure.tight_layout()
    plt.show()

## Decoy control

The real-versus-decoy ipTM gap, per target. A gap near zero means confidence is
uninformative **for that system** - which is not the same as the prediction
being wrong, and is more actionable: it says the number you were about to rely
on cannot tell the right antigen from an unrelated protein.

In [ ]:
decoy = results.reindex(
    columns=["pdb_id", "tier", "confidence_iptm", "confidence_decoy_iptm", "confidence_decoy_gap"]
).rename(
    columns={
        "pdb_id": "target",
        "confidence_iptm": "real ipTM",
        "confidence_decoy_iptm": "decoy ipTM",
        "confidence_decoy_gap": "gap",
    }
)
display(decoy.round(3))

uninformative = decoy[decoy["gap"].fillna(1.0) < confidence.SMALL_DECOY_GAP]
print(
    f"{len(uninformative)} of {len(decoy)} target(s) have a gap under "
    f"{confidence.SMALL_DECOY_GAP}: for those, ipTM is not discriminating."
)

## Verdict

One line on whether confidence tracks accuracy well enough to steer scorer
weights, plus what `recommend_weights` would do with each target.

In [ ]:
best_label, best_rho = max(
    ((label, rho) for label, rho in correlations.items() if rho == rho),
    key=lambda item: abs(item[1]),
    default=("none", float("nan")),
)

if DUMMY:
    verdict = (
        "FIXTURE RUN - no verdict. Run this with a real predictor and a GPU; "
        "what passed here is the code path, not the calibration."
    )
elif best_rho != best_rho:
    verdict = (
        "NO VERDICT: the accuracy axis has no variance or too few honest-tier "
        f"targets (n={len(scoring_rows)}). Widen the selection before acting."
    )
elif abs(best_rho) >= 0.6:
    verdict = (
        f"USABLE: {best_label} tracks {accuracy_column} at rho={best_rho:.2f} "
        f"over {len(scoring_rows)} honest targets - strong enough to steer "
        f"scorer weights."
    )
elif abs(best_rho) >= 0.3:
    verdict = (
        f"WEAK: {best_label} tracks {accuracy_column} at rho={best_rho:.2f}. "
        f"Treat it as a tiebreak, not a gate, and keep the structure-free "
        f"scorer columns weighted."
    )
else:
    verdict = (
        f"UNUSABLE: the best signal is {best_label} at rho={best_rho:.2f}. "
        f"Neither confidence signal predicts accuracy here, so weight the "
        f"structure-free scorers (requires_structure=False) at parity and do "
        f"not let ipTM gate anything."
    )
print(verdict)
print()

for row in results.itertuples():
    recorded = {
        key[len("confidence_"):]: getattr(row, key)
        for key in results.columns
        if key.startswith("confidence_")
    }
    reasons = confidence.weight_reasons(recorded) or ["nothing fired"]
    weights = confidence.recommend_weights(recorded, ["antifold", "gearbind", "esm1v"])
    print(f"{row.pdb_id} ({row.tier}): {weights}")
    for reason in reasons:
        print(f"    - {reason}")